In [2]:
# CELL 1: Setup and Imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import cv2
from pathlib import Path
import time
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import timm

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Set random seed
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
set_seed(42)

# Class names
CLASS_NAMES = ['covid', 'other', 'healthy']

# Create directories
os.makedirs('./models', exist_ok=True)
os.makedirs('./results', exist_ok=True)

Using device: cpu


In [3]:
# CELL 2: Configuration - UPDATE THESE PATHS
CLASSIFICATION_ROOT = r"C:/Users/Administrator/classification"
BLINE_ROOT = r"C:/Users/Administrator/dataset"

IMG_SIZE = 224
BATCH_SIZE = 8
NUM_CLASSES = 3
NUM_SEG_CLASSES = 1
EPOCHS = 5  # Quick comparison
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.01
LAMBDA_SEG = 0.5
CLASS_WEIGHTS = torch.tensor([0.8, 0.9, 1.2], device=device)

print("="*60)
print("CONFIGURATION")
print("="*60)
print(f"Classification root: {CLASSIFICATION_ROOT}")
print(f"B-line root: {BLINE_ROOT}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Epochs: {EPOCHS}")
print(f"Learning rate: {LEARNING_RATE}")

CONFIGURATION
Classification root: C:/Users/Administrator/classification
B-line root: C:/Users/Administrator/dataset
Batch size: 8
Epochs: 5
Learning rate: 0.0001


In [4]:
# CELL 3: Dataset Classes (Same as before)
class ClassificationDataset(Dataset):
    def __init__(self, data_root, split='train', img_size=224):
        self.data_root = Path(data_root)
        self.split = split
        self.img_size = img_size
        self.class_to_idx = {'covid': 0, 'other': 1, 'healthy': 2}
        
        self.images = []
        self.labels = []
        split_path = self.data_root / split
        for class_name, class_idx in self.class_to_idx.items():
            class_path = split_path / class_name
            if class_path.exists():
                for ext in ['*.jpg', '*.jpeg', '*.png']:
                    for img_path in class_path.glob(ext):
                        self.images.append(str(img_path))
                        self.labels.append(class_idx)
        print(f"ClassificationDataset ({split}): {len(self.images)} images")
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = self.images[idx]
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        label = self.labels[idx]
        image = cv2.resize(image, (self.img_size, self.img_size))
        image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
        return image, label


class BlineSegmentationDataset(Dataset):
    def __init__(self, data_root, split='train', has_labels=True, img_size=224):
        self.data_root = Path(data_root)
        self.split = split
        self.has_labels = has_labels
        self.img_size = img_size
        
        split_path = self.data_root / split
        img_dir = split_path / 'images' if (split_path / 'images').exists() else split_path
        self.images_dir = img_dir
        
        if has_labels:
            labels_dir = split_path / 'labels'
            self.has_labels = labels_dir.exists()
            if self.has_labels:
                self.labels_dir = labels_dir
        
        self.image_paths = []
        for ext in ['*.jpg', '*.jpeg', '*.png']:
            self.image_paths.extend(list(self.images_dir.glob(ext)))
        self.image_paths = sorted(self.image_paths)
        
        self.label_paths = []
        if self.has_labels:
            valid = []
            for img_path in self.image_paths:
                label_path = self.labels_dir / f"{img_path.stem}.txt"
                if label_path.exists():
                    valid.append(img_path)
                    self.label_paths.append(label_path)
            self.image_paths = valid
        
        status = "with labels" if self.has_labels else "without labels"
        print(f"BlineSegmentationDataset ({split}): {len(self.image_paths)} images {status}")
    
    def _parse_yolo_polygon(self, label_path, img_h, img_w):
        mask = np.zeros((img_h, img_w), dtype=np.float32)
        try:
            with open(label_path, 'r') as f:
                lines = f.readlines()
            for line in lines:
                line = line.strip()
                if not line or line.startswith('#'):
                    continue
                parts = line.split()
                if len(parts) >= 9:
                    coords = [float(x) for x in parts[1:9]]
                    points = []
                    for i in range(0, len(coords), 2):
                        x = int(coords[i] * img_w)
                        y = int(coords[i+1] * img_h)
                        points.append([x, y])
                    if len(points) >= 3:
                        points = np.array(points, dtype=np.int32)
                        cv2.fillPoly(mask, [points], 1.0)
        except:
            pass
        return mask
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = cv2.imread(str(img_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h, w = image.shape[:2]
        
        if self.has_labels and idx < len(self.label_paths):
            mask = self._parse_yolo_polygon(self.label_paths[idx], h, w)
        else:
            mask = np.zeros((h, w), dtype=np.float32)
        
        image = cv2.resize(image, (self.img_size, self.img_size))
        mask = cv2.resize(mask, (self.img_size, self.img_size), interpolation=cv2.INTER_NEAREST)
        mask = mask[np.newaxis, :, :]
        image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
        mask = torch.from_numpy(mask).float()
        return image, mask


class MultiTaskDataset(Dataset):
    def __init__(self, class_dataset, seg_dataset, is_test=False):
        self.class_dataset = class_dataset
        self.seg_dataset = seg_dataset
        self.is_test = is_test
        self.class_len = len(class_dataset)
        self.seg_len = len(seg_dataset)
    
    def __len__(self):
        return max(self.class_len, self.seg_len)
    
    def __getitem__(self, idx):
        class_idx = idx % self.class_len
        class_img, class_label = self.class_dataset[class_idx]
        seg_idx = idx % self.seg_len
        seg_img, seg_mask = self.seg_dataset[seg_idx]
        return {
            'image': class_img,
            'class_label': class_label,
            'seg_mask': seg_mask,
            'has_seg': torch.tensor(0.0 if self.is_test else 1.0)
        }

In [5]:
# CELL 4: Create DataLoaders
print("\n" + "="*60)
print("CREATING DATALOADERS")
print("="*60)

class_train = ClassificationDataset(CLASSIFICATION_ROOT, split='train')
class_val = ClassificationDataset(CLASSIFICATION_ROOT, split='validation')
class_test = ClassificationDataset(CLASSIFICATION_ROOT, split='test')

seg_train = BlineSegmentationDataset(BLINE_ROOT, split='train', has_labels=True)
seg_val = BlineSegmentationDataset(BLINE_ROOT, split='validation', has_labels=True)
seg_test = BlineSegmentationDataset(BLINE_ROOT, split='test', has_labels=False)

train_mt = MultiTaskDataset(class_train, seg_train, is_test=False)
val_mt = MultiTaskDataset(class_val, seg_val, is_test=False)
test_mt = MultiTaskDataset(class_test, seg_test, is_test=True)

train_loader = DataLoader(train_mt, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_mt, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_mt, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train: {len(train_loader)} batches, Val: {len(val_loader)} batches, Test: {len(test_loader)} batches")


CREATING DATALOADERS
ClassificationDataset (train): 723 images
ClassificationDataset (validation): 198 images
ClassificationDataset (test): 141 images
BlineSegmentationDataset (train): 256 images with labels
BlineSegmentationDataset (validation): 75 images with labels
BlineSegmentationDataset (test): 70 images without labels
Train: 91 batches, Val: 25 batches, Test: 18 batches


In [6]:
# CELL 5: Attention Gate Module (for Models 9, 11)
class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, kernel_size=1, stride=1, padding=0),
            nn.BatchNorm2d(F_int)
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, kernel_size=1, stride=1, padding=0),
            nn.BatchNorm2d(F_int)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, kernel_size=1, stride=1, padding=0),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi

In [22]:
# CELL 6 (COMPLETELY FIXED): Model 6 - ConvNeXt-Tiny + DeepLabV3+
class ASPP(nn.Module):
    def __init__(self, in_channels, out_channels, dilations=[6, 12, 18]):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        
        # All ASPP branches output out_channels
        self.conv1 = nn.Conv2d(in_channels, out_channels, 1, bias=False)
        self.conv2 = nn.Conv2d(in_channels, out_channels, 3, padding=dilations[0], dilation=dilations[0], bias=False)
        self.conv3 = nn.Conv2d(in_channels, out_channels, 3, padding=dilations[1], dilation=dilations[1], bias=False)
        self.conv4 = nn.Conv2d(in_channels, out_channels, 3, padding=dilations[2], dilation=dilations[2], bias=False)
        
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.pool_conv = nn.Conv2d(in_channels, out_channels, 1, bias=False)
        self.pool_bn = nn.BatchNorm2d(out_channels)
        
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.bn3 = nn.BatchNorm2d(out_channels)
        self.bn4 = nn.BatchNorm2d(out_channels)
        
        self.relu = nn.ReLU(inplace=True)
        
    def forward(self, x):
        size = x.shape[2:]
        
        feat1 = self.conv1(x)
        feat1 = self.bn1(feat1)
        feat1 = self.relu(feat1)
        
        feat2 = self.conv2(x)
        feat2 = self.bn2(feat2)
        feat2 = self.relu(feat2)
        
        feat3 = self.conv3(x)
        feat3 = self.bn3(feat3)
        feat3 = self.relu(feat3)
        
        feat4 = self.conv4(x)
        feat4 = self.bn4(feat4)
        feat4 = self.relu(feat4)
        
        feat5 = self.pool(x)
        feat5 = self.pool_conv(feat5)
        feat5 = self.pool_bn(feat5)
        feat5 = self.relu(feat5)
        feat5 = F.interpolate(feat5, size=size, mode='bilinear', align_corners=False)
        
        return torch.cat([feat1, feat2, feat3, feat4, feat5], dim=1)


class DeepLabV3PlusDecoder(nn.Module):
    def __init__(self, encoder_channels, num_classes=1):
        super().__init__()
        # ASPP takes the last encoder channel (768 for ConvNeXt)
        self.aspp = ASPP(encoder_channels[-1], 256)
        # Low-level features from first encoder channel (96 for ConvNeXt)
        self.low_conv = nn.Sequential(
            nn.Conv2d(encoder_channels[0], 48, 1, bias=False),
            nn.BatchNorm2d(48),
            nn.ReLU(inplace=True)
        )
        # Fusion: ASPP outputs 256*5=1280 + low-level 48 = 1328 -> 256
        self.fusion = nn.Sequential(
            nn.Conv2d(256 * 5 + 48, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, num_classes, 1)
        )
    
    def forward(self, features):
        low_level = features[0]      # First encoder output
        high_level = features[-1]    # Last encoder output
        aspp_out = self.aspp(high_level)
        low_proj = self.low_conv(low_level)
        aspp_up = F.interpolate(aspp_out, size=low_proj.shape[2:], mode='bilinear', align_corners=False)
        fused = torch.cat([aspp_up, low_proj], dim=1)
        out = self.fusion(fused)
        out = F.interpolate(out, size=(224, 224), mode='bilinear', align_corners=False)
        return out


class ConvNeXtEncoder(nn.Module):
    def __init__(self, model_name='convnext_tiny', pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, features_only=True)
        self.feature_channels = self.backbone.feature_info.channels()
        print(f"ConvNeXt-Tiny feature channels: {self.feature_channels}")
    
    def forward(self, x):
        return self.backbone(x)


class Model6_ConvNeXt_DeepLabV3(nn.Module):
    def __init__(self, num_classes=3, num_seg_classes=1):
        super().__init__()
        self.encoder = ConvNeXtEncoder('convnext_tiny', pretrained=True)
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(self.encoder.feature_channels[-1], 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
        self.seg_decoder = DeepLabV3PlusDecoder(self.encoder.feature_channels, num_seg_classes)
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
    
    def forward(self, x):
        features = self.encoder(x)
        class_out = self.classifier(features[-1])
        seg_out = self.seg_decoder(features)
        return class_out, seg_out

In [29]:
# CELL 7 (COMPLETELY FIXED): Model 7 - MobileNetV3 + FPN
class MobileNetV3Encoder(nn.Module):
    def __init__(self, model_name='mobilenetv3_large_100', pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, features_only=True)
        self.feature_channels = self.backbone.feature_info.channels()
        print(f"MobileNetV3 feature channels: {self.feature_channels}")
        # MobileNetV3 outputs 5 feature maps: [16, 24, 40, 112, 960]
    
    def forward(self, x):
        return self.backbone(x)


class MobileNetFPNDecoder(nn.Module):
    def __init__(self, encoder_channels, num_classes=1):
        super().__init__()
        # Use all 5 encoder channels
        self.num_levels = len(encoder_channels)
        target_dim = 128
        
        # Lateral connections for each encoder level
        self.lateral = nn.ModuleList()
        for in_ch in encoder_channels:
            self.lateral.append(nn.Conv2d(in_ch, target_dim, kernel_size=1))
        
        # Top-down path (from deepest to shallowest)
        self.topdown = nn.ModuleList()
        for i in range(self.num_levels - 1):
            self.topdown.append(nn.Sequential(
                nn.Conv2d(target_dim, target_dim, 3, padding=1),
                nn.BatchNorm2d(target_dim),
                nn.ReLU(inplace=True)
            ))
        
        # Final prediction
        self.predict = nn.Sequential(
            nn.Conv2d(target_dim, target_dim, 3, padding=1),
            nn.BatchNorm2d(target_dim),
            nn.ReLU(inplace=True),
            nn.Conv2d(target_dim, num_classes, 1)
        )
    
    def forward(self, features):
        # features: [C1, C2, C3, C4, C5] with channels [16, 24, 40, 112, 960]
        # Process from deepest to shallowest
        laterals = [self.lateral[i](feat) for i, feat in enumerate(features)]
        
        # Top-down path starting from deepest (index -1)
        top = laterals[-1]
        
        for i in range(len(laterals) - 2, -1, -1):
            # Upsample top to match next lateral's spatial size
            top = F.interpolate(top, size=laterals[i].shape[2:], mode='bilinear', align_corners=False)
            # Add lateral
            top = top + laterals[i]
            # Apply refinement (skip for the last iteration)
            if i < len(self.topdown):
                top = self.topdown[i](top)
        
        # Final upsampling to 224x224
        top = F.interpolate(top, size=(224, 224), mode='bilinear', align_corners=False)
        out = self.predict(top)
        return out


class Model7_MobileNet_FPN(nn.Module):
    def __init__(self, num_classes=3, num_seg_classes=1):
        super().__init__()
        self.encoder = MobileNetV3Encoder('mobilenetv3_large_100', pretrained=True)
        
        # Classifier using deepest features (960 channels)
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(self.encoder.feature_channels[-1], 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )
        
        # Segmentation decoder
        self.seg_decoder = MobileNetFPNDecoder(self.encoder.feature_channels, num_seg_classes)
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        features = self.encoder(x)
        class_out = self.classifier(features[-1])
        seg_out = self.seg_decoder(features)
        return class_out, seg_out

In [30]:
# CELL 7b: Quick test for Model 7
print("Testing Model 7 forward pass...")
model7_test = Model7_MobileNet_FPN(num_classes=3, num_seg_classes=1)
model7_test.eval()
dummy = torch.randn(2, 3, 224, 224)
try:
    c, s = model7_test(dummy)
    print(f"✅ Model 7 forward pass successful!")
    print(f"   Class output shape: {c.shape}")
    print(f"   Seg output shape: {s.shape}")
except Exception as e:
    print(f"❌ Error: {e}")
    

Testing Model 7 forward pass...


Unexpected keys (classifier.bias, classifier.weight, conv_head.bias, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


MobileNetV3 feature channels: [16, 24, 40, 112, 960]
✅ Model 7 forward pass successful!
   Class output shape: torch.Size([2, 3])
   Seg output shape: torch.Size([2, 1, 224, 224])


In [31]:
# CELL 8: Model 8 - ConvNeXt-Tiny + DeepLabV3+ (same as Model 6)
Model8_ConvNeXt_DeepLabV3 = Model6_ConvNeXt_DeepLabV3

In [44]:
# CELL 9 (FIXED): Model 9 - MobileNetV3 + Attention U-Net with correct output size
class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, kernel_size=1, stride=1, padding=0),
            nn.BatchNorm2d(F_int)
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, kernel_size=1, stride=1, padding=0),
            nn.BatchNorm2d(F_int)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, kernel_size=1, stride=1, padding=0),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi


class AttentionUNetDecoder(nn.Module):
    def __init__(self, encoder_channels, num_classes=1):
        super().__init__()
        # MobileNetV3 channels: [16, 24, 40, 112, 960]
        # Spatial dimensions after encoder: 
        # e1: 112×112, e2: 56×56, e3: 28×28, e4: 14×14, e5: 7×7
        
        # Level 4: 7×7 → 14×14
        self.up4 = nn.ConvTranspose2d(encoder_channels[4], 512, 2, 2)
        self.att4 = AttentionGate(512, encoder_channels[3], 256)
        self.dec4 = self._conv_block(512 + 512, 512)
        
        # Level 3: 14×14 → 28×28
        self.up3 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.att3 = AttentionGate(256, encoder_channels[2], 128)
        self.dec3 = self._conv_block(256 + 256, 256)
        
        # Level 2: 28×28 → 56×56
        self.up2 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.att2 = AttentionGate(128, encoder_channels[1], 64)
        self.dec2 = self._conv_block(128 + 128, 128)
        
        # Level 1: 56×56 → 112×112
        self.up1 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.att1 = AttentionGate(64, encoder_channels[0], 32)
        self.dec1 = self._conv_block(64 + 64, 64)
        
        # Final upsampling to 224×224
        self.final_up = nn.ConvTranspose2d(64, 32, 2, 2)
        self.final_conv = self._conv_block(32, 32)
        self.final = nn.Conv2d(32, num_classes, 1)
    
    def _conv_block(self, in_c, out_c):
        return nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True)
        )
    
    def _project_attention(self, att, target_channels, device):
        """Project attention output to target channels"""
        proj = nn.Conv2d(att.shape[1], target_channels, 1).to(device)
        return proj(att)
    
    def forward(self, features):
        e1, e2, e3, e4, e5 = features[0], features[1], features[2], features[3], features[4]
        device = e1.device
        
        # Decoder Level 4 (7×7 → 14×14)
        d4 = self.up4(e5)  # (B, 512, 14, 14)
        e4_up = F.interpolate(e4, size=d4.shape[2:], mode='bilinear', align_corners=False)
        a4 = self.att4(d4, e4_up)
        # Project attention output to match d4 channels (512)
        a4_proj = nn.Conv2d(112, 512, 1).to(device)(a4)
        d4 = torch.cat([d4, a4_proj], dim=1)  # (B, 1024, 14, 14)
        d4 = self.dec4(d4)  # (B, 512, 14, 14)
        
        # Decoder Level 3 (14×14 → 28×28)
        d3 = self.up3(d4)  # (B, 256, 28, 28)
        e3_up = F.interpolate(e3, size=d3.shape[2:], mode='bilinear', align_corners=False)
        a3 = self.att3(d3, e3_up)
        a3_proj = nn.Conv2d(40, 256, 1).to(device)(a3)
        d3 = torch.cat([d3, a3_proj], dim=1)  # (B, 512, 28, 28)
        d3 = self.dec3(d3)  # (B, 256, 28, 28)
        
        # Decoder Level 2 (28×28 → 56×56)
        d2 = self.up2(d3)  # (B, 128, 56, 56)
        e2_up = F.interpolate(e2, size=d2.shape[2:], mode='bilinear', align_corners=False)
        a2 = self.att2(d2, e2_up)
        a2_proj = nn.Conv2d(24, 128, 1).to(device)(a2)
        d2 = torch.cat([d2, a2_proj], dim=1)  # (B, 256, 56, 56)
        d2 = self.dec2(d2)  # (B, 128, 56, 56)
        
        # Decoder Level 1 (56×56 → 112×112)
        d1 = self.up1(d2)  # (B, 64, 112, 112)
        e1_up = F.interpolate(e1, size=d1.shape[2:], mode='bilinear', align_corners=False)
        a1 = self.att1(d1, e1_up)
        a1_proj = nn.Conv2d(16, 64, 1).to(device)(a1)
        d1 = torch.cat([d1, a1_proj], dim=1)  # (B, 128, 112, 112)
        d1 = self.dec1(d1)  # (B, 64, 112, 112)
        
        # Final upsampling to 224×224
        out = self.final_up(d1)  # (B, 32, 224, 224)
        out = self.final_conv(out)  # (B, 32, 224, 224)
        out = self.final(out)  # (B, 1, 224, 224)
        
        return out


class MobileNetV3Encoder(nn.Module):
    def __init__(self, model_name='mobilenetv3_large_100', pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, features_only=True)
        self.feature_channels = self.backbone.feature_info.channels()
        print(f"MobileNetV3 feature channels: {self.feature_channels}")
    
    def forward(self, x):
        return self.backbone(x)


class Model9_MobileNet_AttentionUNet(nn.Module):
    def __init__(self, num_classes=3, num_seg_classes=1):
        super().__init__()
        self.encoder = MobileNetV3Encoder('mobilenetv3_large_100', pretrained=True)
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(self.encoder.feature_channels[-1], 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )
        self.seg_decoder = AttentionUNetDecoder(self.encoder.feature_channels, num_seg_classes)
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
    
    def forward(self, x):
        features = self.encoder(x)
        class_out = self.classifier(features[-1])
        seg_out = self.seg_decoder(features)
        return class_out, seg_out

In [45]:
# CELL 10: Model 10 - EfficientNet-B0 + Light SegFormer (unchanged)
class EfficientNetB0Encoder(nn.Module):
    def __init__(self, model_name='efficientnet_b0', pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, features_only=True)
        self.feature_channels = self.backbone.feature_info.channels()
        print(f"EfficientNet-B0 feature channels: {self.feature_channels}")
    
    def forward(self, x):
        return self.backbone(x)


class LightSegFormerDecoder(nn.Module):
    def __init__(self, encoder_channels, num_classes=1, decoder_dim=128):
        super().__init__()
        self.proj = nn.ModuleList()
        for in_ch in encoder_channels:
            self.proj.append(
                nn.Sequential(
                    nn.Conv2d(in_ch, decoder_dim, kernel_size=1),
                    nn.BatchNorm2d(decoder_dim),
                    nn.ReLU(inplace=True)
                )
            )
        self.fusion = nn.Sequential(
            nn.Conv2d(decoder_dim * len(encoder_channels), decoder_dim, kernel_size=1),
            nn.BatchNorm2d(decoder_dim),
            nn.ReLU(inplace=True),
            nn.Conv2d(decoder_dim, decoder_dim, kernel_size=3, padding=1),
            nn.BatchNorm2d(decoder_dim),
            nn.ReLU(inplace=True),
            nn.Conv2d(decoder_dim, num_classes, kernel_size=1)
        )
    
    def forward(self, features):
        target_size = features[0].shape[2]
        projected = []
        for i, feat in enumerate(features):
            proj = self.proj[i](feat)
            if proj.shape[2] != target_size:
                proj = F.interpolate(proj, size=(target_size, target_size), 
                                     mode='bilinear', align_corners=False)
            projected.append(proj)
        concat = torch.cat(projected, dim=1)
        out = self.fusion(concat)
        out = F.interpolate(out, size=(224, 224), mode='bilinear', align_corners=False)
        return out


class Model10_EfficientNetB0_SegFormer(nn.Module):
    def __init__(self, num_classes=3, num_seg_classes=1):
        super().__init__()
        self.encoder = EfficientNetB0Encoder('efficientnet_b0', pretrained=True)
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(self.encoder.feature_channels[-1], 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )
        self.seg_decoder = LightSegFormerDecoder(self.encoder.feature_channels, num_seg_classes, decoder_dim=128)
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
    
    def forward(self, x):
        features = self.encoder(x)
        class_out = self.classifier(features[-1])
        seg_out = self.seg_decoder(features)
        return class_out, seg_out

In [53]:
# CELL 11 (FINAL FIX): Model 11 - Swin-Tiny + Simple SegFormer-style Decoder
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

class SwinEncoder(nn.Module):
    def __init__(self, model_name='swin_tiny_patch4_window7_224', pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, features_only=True)
        self.feature_channels = self.backbone.feature_info.channels()
        print(f"Swin-Tiny feature channels: {self.feature_channels}")
    
    def forward(self, x):
        return self.backbone(x)


class SwinSegDecoder(nn.Module):
    def __init__(self, encoder_channels, num_classes=1, decoder_dim=256):
        super().__init__()
        self.proj = nn.ModuleList()
        for ch in encoder_channels:
            self.proj.append(
                nn.Sequential(
                    nn.Conv2d(ch, decoder_dim, kernel_size=1),
                    nn.BatchNorm2d(decoder_dim),
                    nn.ReLU(inplace=True)
                )
            )
        fusion_in = decoder_dim * len(encoder_channels)
        self.fusion = nn.Sequential(
            nn.Conv2d(fusion_in, decoder_dim, kernel_size=1),
            nn.BatchNorm2d(decoder_dim),
            nn.ReLU(inplace=True),
            nn.Conv2d(decoder_dim, decoder_dim, kernel_size=3, padding=1),
            nn.BatchNorm2d(decoder_dim),
            nn.ReLU(inplace=True),
            nn.Conv2d(decoder_dim, num_classes, kernel_size=1)
        )
    
    def forward(self, features):
        target_size = max([f.shape[2] for f in features])
        projected = []
        for i, feat in enumerate(features):
            proj = self.proj[i](feat)
            if proj.shape[2] != target_size:
                proj = F.interpolate(proj, size=(target_size, target_size), 
                                     mode='bilinear', align_corners=False)
            projected.append(proj)
        concat = torch.cat(projected, dim=1)
        out = self.fusion(concat)
        out = F.interpolate(out, size=(224, 224), mode='bilinear', align_corners=False)
        return out


class Model11_Swin_SimpleDecoder(nn.Module):
    def __init__(self, num_classes=3, num_seg_classes=1):
        super().__init__()
        self.encoder = SwinEncoder('swin_tiny_patch4_window7_224', pretrained=True)
        
        # Get the number of features from the last encoder stage
        last_feature_channels = self.encoder.feature_channels[-1]  # 768
        
        # Flatten the spatial dimensions explicitly
        # The last feature map from Swin-Tiny is (B, 768, 7, 7)
        # We need to convert to (B, 768 * 7 * 7) = (B, 37632) or use global pooling
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),     # (B, 768, 7, 7) → (B, 768, 1, 1)
            nn.Flatten(),                 # (B, 768, 1, 1) → (B, 768)
            nn.Linear(last_feature_channels, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
        
        self.seg_decoder = SwinSegDecoder(self.encoder.feature_channels, num_seg_classes)
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        features = self.encoder(x)
        
        # Get deepest features for classification
        deepest = features[-1]  # Shape: (B, 768, H, W)
        
        # Debug print to understand shape (remove in production)
        # print(f"Deepest feature shape: {deepest.shape}")
        
        # Apply global average pooling
        pooled = F.adaptive_avg_pool2d(deepest, (1, 1))  # (B, 768, 1, 1)
        
        # Flatten
        flattened = pooled.view(pooled.size(0), -1)  # (B, 768)
        
        # Classify
        class_out = self.classifier(flattened)  # (B, 3)
        
        # Segment
        seg_out = self.seg_decoder(features)  # (B, 1, 224, 224)
        
        return class_out, seg_out

In [55]:
# CELL 12: Train Models 6-11 (UPDATED)
print("\n" + "="*80)
print("TRAINING MODELS 6-11")
print("="*80)

# Skip Model 6,7,8,9,10,11 - use the fixed Model 9
models_dict = {
    'Model6_ConvNeXt_DeepLabV3': Model6_ConvNeXt_DeepLabV3(num_classes=3, num_seg_classes=1),
    'Model7_MobileNet_FPN': Model7_MobileNet_FPN(num_classes=3, num_seg_classes=1),
    'Model9_MobileNet_AttentionUNet': Model9_MobileNet_AttentionUNet(num_classes=3, num_seg_classes=1),
    'Model10_EfficientNetB0_SegFormer': Model10_EfficientNetB0_SegFormer(num_classes=3, num_seg_classes=1),    
}

results_611 = {}
for name, model in models_dict.items():
    print(f"\n{'='*60}\nTRAINING {name}\n{'='*60}")
    params = sum(p.numel() for p in model.parameters())
    print(f"Parameters: {params:,} ({params * 4 / 1024**2:.1f} MB)")
    results_611[name] = train_model(model, train_loader, val_loader, name, epochs=EPOCHS)


TRAINING MODELS 6-11
ConvNeXt-Tiny feature channels: [96, 192, 384, 768]


Unexpected keys (classifier.bias, classifier.weight, conv_head.bias, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


MobileNetV3 feature channels: [16, 24, 40, 112, 960]


Unexpected keys (classifier.bias, classifier.weight, conv_head.bias, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


MobileNetV3 feature channels: [16, 24, 40, 112, 960]


...

Model10_EfficientNetB0_SegFormer - E3: Loss=4.1604, Acc=46.46%, Dice=0.3223


Model10_EfficientNetB0_SegFormer E4: 100%|████████████████████████████████| 91/91 [04:43<00:00,  3.11s/it, loss=2.0892]


Model10_EfficientNetB0_SegFormer - E4: Loss=3.5110, Acc=52.02%, Dice=0.3141


Model10_EfficientNetB0_SegFormer E5: 100%|████████████████████████████████| 91/91 [04:51<00:00,  3.20s/it, loss=3.4317]


Model10_EfficientNetB0_SegFormer - E5: Loss=2.9854, Acc=52.02%, Dice=0.3383


In [56]:
# CELL 14: Results Summary
print("\n" + "="*80)
print("TRAINING RESULTS SUMMARY")
print("="*80)

for name, results in results_611.items():
    final_acc = results['val_acc'][-1] if results['val_acc'] else 0
    final_dice = results['val_dice'][-1] if results['val_dice'] else 0
    print(f"{name}: Acc={final_acc:.2f}%, Dice={final_dice:.4f}")


TRAINING RESULTS SUMMARY
Model6_ConvNeXt_DeepLabV3: Acc=61.11%, Dice=0.3596
Model7_MobileNet_FPN: Acc=64.65%, Dice=0.3392
Model9_MobileNet_AttentionUNet: Acc=52.53%, Dice=0.0012
Model10_EfficientNetB0_SegFormer: Acc=52.02%, Dice=0.3383


In [57]:
# CELL 15: Evaluate Models 6-11 on Test Set
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Ensure we have test_loader
if 'test_loader' not in locals():
    print("test_loader not found. Please load test data first.")
else:
    print(f"Test loader has {len(test_loader.dataset)} images.")

# Dictionary to store test results
test_results = {}

# Function to evaluate a model on test set
def evaluate_on_test(model, test_loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    dice_scores = []
    
    with torch.no_grad():
        for batch in test_loader:
            images = batch['image'].to(device)
            labels = batch['class_label'].cpu().numpy()
            seg_masks = batch['seg_mask'].numpy()
            has_seg = batch['has_seg'].numpy()
            
            class_logits, seg_logits = model(images)
            probs = torch.softmax(class_logits, dim=1)
            preds = torch.argmax(probs, dim=1).cpu().numpy()
            
            all_preds.extend(preds)
            all_labels.extend(labels)
            all_probs.extend(probs.cpu().numpy())
            
            # Dice for images with segmentation masks
            seg_probs = torch.sigmoid(seg_logits).cpu().numpy()
            for i in range(len(images)):
                if has_seg[i] > 0:
                    pred_mask = (seg_probs[i, 0] > 0.5).astype(np.float32)
                    true_mask = seg_masks[i, 0]
                    intersection = (pred_mask * true_mask).sum()
                    dice = (2. * intersection) / (pred_mask.sum() + true_mask.sum() + 1e-6)
                    dice_scores.append(dice)
    
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
    recall = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
    f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    avg_dice = np.mean(dice_scores) if dice_scores else 0.0
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'dice': avg_dice,
        'predictions': all_preds,
        'labels': all_labels,
        'probabilities': all_probs
    }

# If we have trained models in results_611, we need the model objects themselves.
# Assuming we have saved the models in a dictionary 'trained_models' during training.
# If not, we need to re-instantiate and load weights. Here we'll assume the models are already in memory.
# For safety, we'll re-create models and load state dicts if saved.

# Check if we have model objects in a dictionary; otherwise, recreate.
if 'trained_models' in locals():
    models_dict = trained_models
else:
    # Recreate model instances (assuming they are defined in previous cells)
    models_dict = {
        'Model6_ConvNeXt_DeepLabV3': Model6_ConvNeXt_DeepLabV3(num_classes=3, num_seg_classes=1),
        'Model7_MobileNet_FPN': Model7_MobileNet_FPN(num_classes=3, num_seg_classes=1),
        'Model9_MobileNet_AttentionUNet': Model9_MobileNet_AttentionUNet(num_classes=3, num_seg_classes=1),
        'Model10_EfficientNetB0_SegFormer': Model10_EfficientNetB0_SegFormer(num_classes=3, num_seg_classes=1),
        
    }
    # Load best weights if available
    for name, model in models_dict.items():
        weight_path = f'./models/{name}_best.pth'
        if os.path.exists(weight_path):
            model.load_state_dict(torch.load(weight_path, map_location=device))
            print(f"Loaded {name} from {weight_path}")
        else:
            print(f"Warning: No saved weights for {name}. Using random initialization.")

# Evaluate each model
for name, model in models_dict.items():
    print(f"\nEvaluating {name}...")
    model.to(device)
    model.eval()
    res = evaluate_on_test(model, test_loader, device)
    test_results[name] = res
    print(f"  Accuracy: {res['accuracy']*100:.2f}%")
    print(f"  Precision: {res['precision']*100:.2f}%")
    print(f"  Recall: {res['recall']*100:.2f}%")
    print(f"  F1: {res['f1']*100:.2f}%")
    print(f"  Dice (segmentation): {res['dice']:.4f}")

Test loader has 141 images.
ConvNeXt-Tiny feature channels: [96, 192, 384, 768]


Unexpected keys (classifier.bias, classifier.weight, conv_head.bias, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


MobileNetV3 feature channels: [16, 24, 40, 112, 960]


Unexpected keys (classifier.bias, classifier.weight, conv_head.bias, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


MobileNetV3 feature channels: [16, 24, 40, 112, 960]


Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


EfficientNet-B0 feature channels: [16, 24, 40, 112, 320]

Evaluating Model6_ConvNeXt_DeepLabV3...
  Accuracy: 39.01%
  Precision: 15.22%
  Recall: 39.01%
  F1: 21.89%
  Dice (segmentation): 0.0000

Evaluating Model7_MobileNet_FPN...
  Accuracy: 39.01%
  Precision: 15.22%
  Recall: 39.01%
  F1: 21.89%
  Dice (segmentation): 0.0000

Evaluating Model9_MobileNet_AttentionUNet...
  Accuracy: 21.99%
  Precision: 4.83%
  Recall: 21.99%
  F1: 7.93%
  Dice (segmentation): 0.0000

Evaluating Model10_EfficientNetB0_SegFormer...
  Accuracy: 21.99%
  Precision: 4.83%
  Recall: 21.99%
  F1: 7.93%
  Dice (segmentation): 0.0000


In [58]:
# CELL 16: Create Comparison Table
print("\n" + "="*80)
print("FINAL TEST PERFORMANCE COMPARISON (MODELS 6-11)")
print("="*80)

comparison_data = []
for name, res in test_results.items():
    comparison_data.append({
        'Model': name,
        'Accuracy (%)': f"{res['accuracy']*100:.2f}",
        'Precision (%)': f"{res['precision']*100:.2f}",
        'Recall (%)': f"{res['recall']*100:.2f}",
        'F1 (%)': f"{res['f1']*100:.2f}",
        'Dice': f"{res['dice']:.4f}"
    })

df_compare = pd.DataFrame(comparison_data)
print(df_compare.to_string(index=False))

# Save to CSV
df_compare.to_csv('./results/models_6_11_test_performance.csv', index=False)
print("\n✅ Saved to ./results/models_6_11_test_performance.csv")


FINAL TEST PERFORMANCE COMPARISON (MODELS 6-11)
                           Model Accuracy (%) Precision (%) Recall (%) F1 (%)   Dice
       Model6_ConvNeXt_DeepLabV3        39.01         15.22      39.01  21.89 0.0000
            Model7_MobileNet_FPN        39.01         15.22      39.01  21.89 0.0000
  Model9_MobileNet_AttentionUNet        21.99          4.83      21.99   7.93 0.0000
Model10_EfficientNetB0_SegFormer        21.99          4.83      21.99   7.93 0.0000

✅ Saved to ./results/models_6_11_test_performance.csv


In [59]:
# CELL 17: Confusion Matrices for Each Model
import itertools

def plot_confusion_matrix(labels, preds, model_name, class_names=['COVID-19', 'Other Disease', 'Healthy']):
    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(6, 5))
    plt.imshow(cm, interpolation='nearest', cmap='Blues')
    plt.title(f'Confusion Matrix - {model_name}', fontweight='bold')
    plt.colorbar()
    tick_marks = np.arange(len(class_names))
    plt.xticks(tick_marks, class_names, rotation=45)
    plt.yticks(tick_marks, class_names)
    
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], 'd'),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(f'./results/cm_{model_name}.png', dpi=150)
    plt.close()
    print(f"Confusion matrix saved for {model_name}")

for name, res in test_results.items():
    plot_confusion_matrix(res['labels'], res['predictions'], name)

Confusion matrix saved for Model6_ConvNeXt_DeepLabV3
Confusion matrix saved for Model7_MobileNet_FPN
Confusion matrix saved for Model9_MobileNet_AttentionUNet
Confusion matrix saved for Model10_EfficientNetB0_SegFormer
